In [1]:
# Set working directory
import os
import pathlib
working_dir = '/Users/joc0445/Library/CloudStorage/OneDrive-HarvardUniversity/VS/Ecuador/Trade'
os.chdir(working_dir)
os.makedirs('data/intermediate', exist_ok=True)

# Import libraries
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import operator
import pandas as pd
import seaborn as sns
import plotly.express as px
import plotly.offline as pyo
from plotly.graph_objects import Frame
import plotly.graph_objects as go

# Ecuador Distances

In [24]:
# Import the file data/input/Gravity_V202211.dta
df = pd.read_stata('data/input/Gravity_V202211.dta')

# Keep if iso3_o == 'ECU'
df = df[df['iso3_o'] == 'ECU']
# Keep if year = 2021
df = df[df['year'] == 2021]
# Drop if iso3_d == 'ECU'
df = df[df['iso3_d'] != 'ECU']
# Keep if country_exists_d == 1
df = df[df['country_exists_d'] == 1]
# Keep unique values of iso3_d
df = df.drop_duplicates(subset=['iso3_d'])

# Keep iso3_d and dist
df = df[['iso3_d', 'dist']]

# Rename iso3_d to iso3 and dist to distance
df.to_csv('data/intermediate/ecuador_distance.csv', index=False)

df.head()

,iso3_d,dist
1212193,ABW,1957.0
1212267,AFG,15209.0
1212341,AGO,10326.0
1212415,AIA,2914.0
1212489,ALB,10982.0


# Bilateral Distances

In [25]:
# Import the file data/input/Gravity_V202211.dta
df = pd.read_stata('data/input/Gravity_V202211.dta')

# Keep if year is 2021
df = df[df['year'] == 2021]
# Keep if iso3_o != iso3_d
df = df[df['iso3_o'] != df['iso3_d']]
# Keep if country_exists_d == 1
df = df[df['country_exists_d'] == 1]
# Keep if country_exists_o == 1
df = df[df['country_exists_o'] == 1]
# Keep iso3_o, iso3_d, and dist
df = df[['iso3_o', 'iso3_d', 'dist']]

# Export the dataframe to a csv file
df.to_csv('data/intermediate/bilateral_distances.csv', index=False)

df.head()

,iso3_o,iso3_d,dist
147,ABW,AFG,13256.0
221,ABW,AGO,9505.0
295,ABW,AIA,978.0
369,ABW,ALB,9090.0
443,ABW,AND,7570.0


# Countries

In [26]:
# Import the rankings.csv file
countries = pd.read_csv('data/input/rankings.csv')

# Keep only countries in rankings year 2023
countries = countries[countries['year'] == 2023]

# Rename country_iso3_code to iso3
countries = countries.rename(columns={'country_iso3_code': 'iso3'})

# Keep iso3
countries = countries[['iso3']]

# Export the dataframe to a csv file
countries.to_csv('data/intermediate/countries.csv', index=False)

countries.head()

,iso3
28,AFG
58,ALB
88,DZA
118,AGO
148,AZE


# Product Trade

In [27]:
# Import the hs92_country_country_product_year_6_2020_2024.csv file
product = pd.read_csv('data/input/hs92_country_country_product_year_6_2020_2024.csv')

# Create a hs92 column with the first 4 characters of the product_hs92_code column
product['hs92'] = product['product_hs92_code'].str[:4]

# Group by year and hs92 and sum the export_value column
product = product.groupby(['year', 'hs92'])['export_value'].sum().reset_index()

# Keep if year 2020 or 2024
product = product[product['year'].isin([2020, 2024])]

# Pivotwide the dataframe to have years as columns and hs92 as index
product = product.pivot(index='hs92', columns='year', values='export_value').reset_index()

# Calculate 5yr_growth = (export_value_2024 - export_value_2020) / export_value_2020
product['5yr_growth'] = (product[2024] - product[2020]) / product[2020]

# Create above_median dummy variable where 1 if 5yr_growth is above the median and 0 otherwise
median_growth = product['5yr_growth'].median()
product['above_median'] = (product['5yr_growth'] > median_growth).astype(int)

# Keep only hs92, 5yr_growth and above_median columns
product = product[['hs92', '5yr_growth', 'above_median']]

# Export the dataframe to a csv file
product.to_csv('data/intermediate/hs92_growth.csv', index=False)

product.head()

/var/folders/hw/hg60dm0x0971qftcm4gzwp980000gq/T/ipykernel_1979/1617415178.py:2: DtypeWarning:

Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.



year,hs92,5yr_growth,above_median
0,0101,0.417620,1
1,0102,0.180656,0
2,0103,0.108275,0
3,0104,1.329122,1
4,0105,0.241571,0


# Bilateral trade data

In [28]:
# Import the hs92_country_country_product_year_6_2020_2024.csv file
df = pd.read_csv('data/input/hs92_country_country_product_year_6_2020_2024.csv')

# Create a hs92 column with the first 4 characters of the product_hs92_code column
df['hs92'] = df['product_hs92_code'].str[:4]

# Drrop if hs92 is 9999 or XXXX
df = df[~df['hs92'].isin(['9999', 'XXXX'])]

# Group by year, country_iso3_code, partner_iso3_code and hs92 and sum the export_value column
df = df.groupby(['year', 'country_iso3_code', 'partner_iso3_code', 'hs92'])['export_value'].sum().reset_index()

# Rename country_iso3_code to iso3_o and partner_iso3_code to iso3_d
df = df.rename(columns={'country_iso3_code': 'iso3_o', 'partner_iso3_code': 'iso3_d'})

# Group by iso3_o, iso3_d and hs92 and take the mean of the export_value column
df = df.groupby(['iso3_o', 'iso3_d', 'hs92'])['export_value'].mean().reset_index()

df.head()

/var/folders/hw/hg60dm0x0971qftcm4gzwp980000gq/T/ipykernel_1979/2830624519.py:2: DtypeWarning:

Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.



,iso3_o,iso3_d,hs92,export_value
0,AFG,AGO,0713,32665.0
1,AFG,AGO,2522,21010.0
2,AFG,AGO,3823,47679.0
3,AFG,AGO,6908,2701.0
4,AFG,AGO,8438,2505.0


In [29]:
# Import the countries.csv file
countries = pd.read_csv('data/intermediate/countries.csv')

# Import the bilateral_distances.csv file
distances = pd.read_csv('data/intermediate/bilateral_distances.csv')


# Keep only bilateral flows where both countries are in rankings year 2023 list
df = df[df['iso3_o'].isin(countries['iso3']) & df['iso3_d'].isin(countries['iso3'])]


# Merge the df and distances dataframes on iso3_o and iso3_d
df = pd.merge(df, distances, left_on=['iso3_o', 'iso3_d'], right_on=['iso3_o', 'iso3_d'], how='left')

df.head()

,iso3_o,iso3_d,hs92,export_value,dist
0,AFG,AGO,0713,32665.0,7593.0
1,AFG,AGO,2522,21010.0,7593.0
2,AFG,AGO,3823,47679.0,7593.0
3,AFG,AGO,6908,2701.0,7593.0
4,AFG,AGO,8438,2505.0,7593.0


In [30]:
# Effective Number of Exporters and Travelled Distance by HS92

# 1. Pre-calculate helper columns for aggregation
# Square of export value is needed for HHI
df['export_sq'] = df['export_value'] ** 2

# Distance weighted by export value is needed for the weighted average
df['dist_x_val'] = df['dist'] * df['export_value']

# 2. Group by 'hs92' and aggregate
# We sum the components needed for the final formulas
grouped = df.groupby('hs92').agg(
    total_export=('export_value', 'sum'),
    sum_sq_export=('export_sq', 'sum'),
    sum_weighted_dist=('dist_x_val', 'sum')
)

# 3. Calculate the final metrics
# Effective Number of Exporters = 1 / HHI
# HHI = Sum( (x_i / Total_x)^2 ) = Sum(x_i^2) / (Total_x)^2
# Therefore: Inverse HHI = (Total_x)^2 / Sum(x_i^2)
grouped['eff_num_exp'] = (grouped['total_export'] ** 2) / grouped['sum_sq_export']

# Weighted Average Distance = Sum(dist * val) / Sum(val)
grouped['travelled_distance'] = grouped['sum_weighted_dist'] / grouped['total_export']

# Round the results to 2 decimal places
grouped['eff_num_exp'] = grouped['eff_num_exp'].round(2)
grouped['travelled_distance'] = grouped['travelled_distance'].round(2)

# 4. Final cleanup
# Reset index to make 'hs92' a column again and select only requested columns
result_df = grouped[['eff_num_exp', 'travelled_distance']].reset_index()

# Merge the result_df with the product dataframe to get the 5yr_growth and above_median columns
result_df = result_df.merge(product, on='hs92', how='left')

# Import HS92 4-digit reference data (instead of opportunities)
hs92_ref = pd.read_csv('data/input/hs92_4digits.csv', encoding='utf-8-sig')
hs92_ref['hs92'] = hs92_ref['product_hs92_code'].astype(str).str.zfill(4)
hs92_ref = hs92_ref[['hs92', 'product_name_short', 'sector', 'green_product']]
# Drop if hs92 is 9999 or XXXX
hs92_ref = hs92_ref[~hs92_ref['hs92'].isin(['9999', 'XXXX'])]

# Merge product descriptors into final result
result_df = result_df.merge(hs92_ref, on='hs92', how='left')

# Export the result to a CSV file
result_df.to_csv('data/intermediate/hs92_attributes.csv', index=False)

result_df.head() 


,hs92,eff_num_exp,travelled_distance,5yr_growth,above_median,product_name_short,sector,green_product
0,0101,12.09,3245.08,0.417620,1,Horses,Agriculture,False
1,0102,17.92,2831.28,0.180656,0,Bovine,Agriculture,False
2,0103,12.10,933.95,0.108275,0,Swine,Agriculture,False
3,0104,6.26,1623.25,1.329122,1,Sheep,Agriculture,False
4,0105,21.37,1648.53,0.241571,0,Fowl,Agriculture,False


In [9]:
# Clear working environment
del df, product, countries, grouped, hs92_ref

# For distances create a iso3_o_iso3_d column
distances['iso3_o_iso3_d'] = distances['iso3_o'] + '_' + distances['iso3_d']

distances.head()

,iso3_o,iso3_d,dist,iso3_o_iso3_d
0,ABW,AFG,13256.0,ABW_AFG
1,ABW,AGO,9505.0,ABW_AGO
2,ABW,AIA,978.0,ABW_AIA
3,ABW,ALB,9090.0,ABW_ALB
4,ABW,AND,7570.0,ABW_AND


In [18]:
# Identification of Potential Market (Equal or Below travelled distance)
# Import the hs92_country_country_product_year_6_2020_2024.csv file
df = pd.read_csv('data/input/hs92_country_country_product_year_6_2020_2024.csv')

# Create a hs92 column with the first 4 characters of the product_hs92_code column
df['hs92'] = df['product_hs92_code'].str[:4]

# Drrop if hs92 is 9999 or XXXX
df = df[~df['hs92'].isin(['9999', 'XXXX'])]

# Merge with result_df to get the travelled_distance column
df = df.merge(result_df[['hs92', 'travelled_distance']], on='hs92', how='left')

# Rename country_iso3_code to iso3_o and partner_iso3_code to iso3_d
df = df.rename(columns={'country_iso3_code': 'iso3_o', 'partner_iso3_code': 'iso3_d'})

# Create a iso3_o_iso3_d column
df['iso3_o_iso3_d'] = df['iso3_o'] + '_' + df['iso3_d']

# Merge with distances dataframe to get the distance between iso3_o and iso3_d
df = df.merge(distances[['iso3_o_iso3_d', 'dist']], on='iso3_o_iso3_d', how='left')

# Create a dummy variable potential_market where 1 if dist is less than or equal to travelled_distance and 0 otherwise
df['potential_market'] = (df['dist'] <= df['travelled_distance']).astype(int)

df.head()

/var/folders/hw/hg60dm0x0971qftcm4gzwp980000gq/T/ipykernel_1979/1427254281.py:3: DtypeWarning:

Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.



,country_id,iso3_o,partner_country_id,iso3_d,product_id,product_hs92_code,year,export_value,import_value,hs92,travelled_distance,iso3_o_iso3_d,dist,potential_market
0,4,AFG,24,AGO,6627,382390,2021,47679,0,3823,4127.51,AFG_AGO,7593.0,0
1,4,AFG,24,AGO,8051,690890,2021,2701,0,6908,3992.10,AFG_AGO,7593.0,0
2,4,AFG,24,AGO,9456,853669,2021,4370,0,8536,4558.98,AFG_AGO,7593.0,0
3,4,AFG,24,AGO,9543,860900,2021,1720,0,8609,5924.56,AFG_AGO,7593.0,0
4,4,AFG,24,AGO,9766,902610,2021,1659,0,9026,5315.14,AFG_AGO,7593.0,0


In [ ]:
# Keep potential-market map by year (not only 2024)
potential_markets = df[['year', 'iso3_o', 'iso3_d', 'hs92', 'travelled_distance', 'dist', 'potential_market']].drop_duplicates().copy()

potential_markets.head()


In [ ]:
# Compute potential market imports by country-product-year
df = df.copy()

# Total imports by destination/product/year
total_imports = df.groupby(['year', 'iso3_d', 'hs92'])['export_value'].sum().reset_index(name='total_imports')

# Merge total imports into potential-market map
potential_markets = potential_markets.merge(total_imports, on=['year', 'iso3_d', 'hs92'], how='left')
potential_markets['total_imports'] = pd.to_numeric(potential_markets['total_imports'], errors='coerce').fillna(0.0)

# Potential market imports = total imports if destination is within travelled distance, else 0
potential_markets['potential_market_imports'] = potential_markets['total_imports'] * potential_markets['potential_market']

# Main year-level output
potential_markets.to_csv('data/intermediate/potential_market_by_country_product_year.csv', index=False)

# Legacy compatibility output (2024-only)
potential_markets_2024 = potential_markets[potential_markets['year'] == 2024].copy()
potential_markets_2024.to_csv('data/intermediate/potential_market_by_country_product.csv', index=False)

potential_markets.head()


,iso3_o,iso3_d,hs92,travelled_distance,dist,potential_market,total_imports,potential_market_imports
0,AFG,AZE,1502,5373.45,1821.0,1,1158575.0,1158575.0
1,AFG,AZE,1704,3263.68,1821.0,1,46680894.0,46680894.0
2,AFG,AZE,1806,2446.37,1821.0,1,107860027.0,107860027.0
3,AFG,AZE,1905,2328.93,1821.0,1,114624365.0,114624365.0
4,AFG,AZE,2526,3655.80,1821.0,1,134744.0,134744.0


In [ ]:
# Aggregate potential market by product and compute potential-market growth (2020-2024)
potential_market_by_product_year = (
    potential_markets
    .groupby(['iso3_d', 'hs92', 'year'], as_index=False)['potential_market_imports']
    .sum()
    .rename(columns={'potential_market_imports': 'potential_market_size'})
)

# Save long yearly series
potential_market_by_product_year.to_csv('data/intermediate/potential_market_by_product_year.csv', index=False)

# Legacy compatibility output expected by some routines (2024, column name potential_market_imports_sum)
potential_market_by_product_2024 = potential_market_by_product_year[potential_market_by_product_year['year'] == 2024].copy()
potential_market_by_product_2024 = potential_market_by_product_2024.rename(columns={'potential_market_size': 'potential_market_imports_sum'})
potential_market_by_product_2024[['iso3_d', 'hs92', 'potential_market_imports_sum']].to_csv('data/intermediate/potential_market_by_product.csv', index=False)

# Build 2020 vs 2024 potential market CAGR by destination country and product
pm_pivot = potential_market_by_product_year.pivot_table(index=['iso3_d', 'hs92'], columns='year', values='potential_market_size', aggfunc='sum', fill_value=0).reset_index()
if 2020 not in pm_pivot.columns:
    pm_pivot[2020] = 0.0
if 2024 not in pm_pivot.columns:
    pm_pivot[2024] = 0.0

pm_pivot['potential_market_growth_5y'] = np.where(
    (pm_pivot[2020] > 0) & (pm_pivot[2024] > 0),
    (pm_pivot[2024] / pm_pivot[2020]) ** (1/5) - 1,
    0.0
)

potential_market_growth = pm_pivot[['iso3_d', 'hs92', 2020, 2024, 'potential_market_growth_5y']].rename(columns={2020: 'potential_market_size_2020', 2024: 'potential_market_size_2024'})
potential_market_growth.to_csv('data/intermediate/potential_market_growth_by_product.csv', index=False)

potential_market_growth.head()


,iso3_d,hs92,potential_market_imports_sum
0,ABW,0105,121661.0
1,ABW,0106,134520.0
2,ABW,0201,6107082.0
3,ABW,0202,39545664.0
4,ABW,0203,4332036.0


In [3]:
# Unilateral 4-digit trade matrix and economic complexity inputs
from ecomplexity import ecomplexity

# Keep only HS92 4-digit codes present in the reference file
hs92_ref = pd.read_csv('data/input/hs92_4digits.csv', encoding='utf-8-sig')
valid_hs92 = set(hs92_ref['product_hs92_code'].astype(str).str.zfill(4).unique())

# Keep only countries in rankings year 2023
rankings = pd.read_csv('data/input/rankings.csv')
valid_countries = set(rankings.loc[rankings['year'] == 2023, 'country_iso3_code'].astype(str).unique())

# Build unilateral exports: sum exports over all destinations by exporter-year-product (HS92 4-digit)
trade = pd.read_csv('data/input/hs92_country_country_product_year_6_2020_2024.csv')
trade['product'] = trade['product_hs92_code'].astype(str).str[:4].str.zfill(4)
trade = trade[trade['product'].isin(valid_hs92)].copy()
trade = trade[trade['country_iso3_code'].isin(valid_countries)].copy()

trade = (
    trade.groupby(['year', 'country_iso3_code', 'product'], as_index=False)['export_value']
    .sum()
    .rename(columns={'year': 'time', 'country_iso3_code': 'location', 'export_value': 'value'})
)

def calculate_rca_custom(data: pd.DataFrame) -> pd.DataFrame:
    df = data.copy()

    country_total = df.groupby(['time', 'location'])['value'].transform('sum')
    product_total = df.groupby(['time', 'product'])['value'].transform('sum')
    world_total = df.groupby(['time'])['value'].transform('sum')

    df['rca'] = (df['value'] / country_total) / (product_total / world_total)
    df['rca'] = df['rca'].replace([np.inf, -np.inf], np.nan).fillna(0)

    # Bounded transform for manual MCP in [0,1): avoids negative/undefined values
    df['rca_transformation'] = df['rca'] / (1 + df['rca'])
    df['rca_transformation'] = df['rca_transformation'].replace([np.inf, -np.inf], 0).fillna(0).clip(0, 1)

    return df

trade_ready = calculate_rca_custom(trade)

# Pass only required columns so ecomplexity doesn't produce duplicate rca_x/rca_y fields
manual_input = trade_ready[['time', 'location', 'product', 'rca_transformation']].rename(
    columns={'rca_transformation': 'value'}
)

trade_cols = {
    'time': 'time',
    'loc': 'location',
    'prod': 'product',
    'val': 'value'
}

cdata_con = ecomplexity(manual_input, trade_cols, presence_test='manual')

# Keep all countries in output (file name retained for compatibility)
complexity_all = cdata_con.copy()
if 'value' in complexity_all.columns:
    complexity_all = complexity_all.rename(columns={'value': 'rca_transformation'})

# Create a density_percentile column by year and product using the rank of density within each year and product group
complexity_all['density_percentile'] = complexity_all.groupby(['time', 'product'])['density'].rank(pct=True)

complexity_all.to_csv('data/intermediate/complexity_calculations.csv', index=False)

complexity_all.head()



/var/folders/hw/hg60dm0x0971qftcm4gzwp980000gq/T/ipykernel_3835/2467834599.py:13: DtypeWarning:

Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.



2020
Percentage of pairs compared that meet log-supermodularity condition: 37.08%
2021


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/ecomplexity/ecomplexity.py:252: UserWarning:

Year 2020: Log-supermodularity condition is not fully satisfied (37.08% of pairs compared satisfy this condition). The ECI and PCI values may not be a true representation of the complexity. More details at: https://growthlab.hks.harvard.edu/publications/structural-ranking-economic-complexity



Percentage of pairs compared that meet log-supermodularity condition: 20.59%
2022


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/ecomplexity/ecomplexity.py:252: UserWarning:

Year 2021: Log-supermodularity condition is not fully satisfied (20.59% of pairs compared satisfy this condition). The ECI and PCI values may not be a true representation of the complexity. More details at: https://growthlab.hks.harvard.edu/publications/structural-ranking-economic-complexity



...

Percentage of pairs compared that meet log-supermodularity condition: 14.67%
2024


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/ecomplexity/ecomplexity.py:252: UserWarning:

Year 2023: Log-supermodularity condition is not fully satisfied (14.67% of pairs compared satisfy this condition). The ECI and PCI values may not be a true representation of the complexity. More details at: https://growthlab.hks.harvard.edu/publications/structural-ranking-economic-complexity



Percentage of pairs compared that meet log-supermodularity condition: 18.76%


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/ecomplexity/ecomplexity.py:252: UserWarning:

Year 2024: Log-supermodularity condition is not fully satisfied (18.76% of pairs compared satisfy this condition). The ECI and PCI values may not be a true representation of the complexity. More details at: https://growthlab.hks.harvard.edu/publications/structural-ranking-economic-complexity



,location,product,rca_transformation,time,diversity,ubiquity,mcp,eci,pci,density,coi,cog,rca,density_percentile
0,AFG,0101,0.000000,2020,82.187225,18.539694,0.000000,-1.456416,1.358391,0.060159,-1.063394,0.617384,0.000000,0.220690
1,AFG,0102,0.000000,2020,82.187225,34.157039,0.000000,-1.456416,-1.451560,0.074990,-1.063394,0.244106,0.000000,0.220690
2,AFG,0103,0.000000,2020,82.187225,15.274325,0.000000,-1.456416,2.340966,0.052837,-1.063394,0.674624,0.000000,0.213793
3,AFG,0104,0.713292,2020,82.187225,32.449797,0.713292,-1.456416,-2.719490,0.091985,-1.063394,0.011818,0.713292,0.220690
4,AFG,0105,0.000000,2020,82.187225,33.290177,0.000000,-1.456416,-0.233855,0.070435,-1.063394,0.339559,0.000000,0.200000


In [ ]:
# Precompute V1 dashboard metrics (HS4) for ECU and ARG
import subprocess
import textwrap
from pathlib import Path

root = Path.cwd()
if (root / 'ecuador_opportunities').exists() and (root / 'ARG' / 'V1_ARG').exists():
    code_root = root
elif (root / 'code' / 'ecuador_opportunities').exists() and (root / 'code' / 'ARG' / 'V1_ARG').exists():
    code_root = root / 'code'
else:
    code_root = root

venv_python = code_root / 'ecuador_opportunities' / '.venv' / 'bin' / 'python'
if not venv_python.exists():
    raise FileNotFoundError(f'Python env not found: {venv_python}')

script = textwrap.dedent(f"""
import importlib.util
from pathlib import Path
code_root = Path(r'{code_root}')
for rel, country in [('ecuador_opportunities', 'ECU'), ('ARG/V1_ARG', 'ARG')]:
    module_dir = code_root / rel
    module_file = module_dir / 'data_utils.py'
    spec = importlib.util.spec_from_file_location('data_utils_' + country.lower(), module_file)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    ref = mod.load_hs92_reference()
    out = mod.load_or_build_v1_hs4_metrics(ref['hs4'].tolist(), year=2024)
    print(country, out.shape, mod.intermediate_dir())
""")

subprocess.run([str(venv_python), '-c', script], check=True)
